# 🏨 Gran Canaria – Tetthetsanalyse av overnattingssteder med H3-hexagoner

**Hva skal vi gjøre?** Vi henter overnattingsdata direkte fra skyen med SQL, teller hvor mange steder som ligger i hver sekskantede rute i et rutenett over øya, og tegner resultatet som et fargelagt tetthetskart. Til slutt eksporterer vi dataene slik at du kan bygge videre på dem med kodeagenter for å lage kule webapps på egenhånd.

Vi bruker:

- **Overture Maps** (theme=places) som datakilde – og finner **nyeste versjon automatisk** via Overture sin STAC-katalog
- **DuckDB** for effektive sky-native SQL-spørringer med predicate pushdown
- **H3** (Ubers sekskantede rutenett) for tetthetsaggregering
- **Folium** for interaktiv kartvisualisering

<a target="_blank" href="https://colab.research.google.com/github/kartAI/skygeo/blob/main/src/skygeo-workshop/gran_canaria_overnatting_tetthet.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

### 🧭 Slik er workshopen bygget opp

| Steg | Hva skjer |
|---|---|
| 1 | Oppsett – installer pakker, koble til data i skyen |
| 2 | Definer studieområdet (bounding box) |
| 3 | Hent overnattingssteder fra Overture Maps |
| 4 | Aggreger til H3-tetthet og visualiser på kart |
| 5 | Eksporter data og bygg videre med et eget kart |

> 💡 **Tips:** Kjør cellene i rekkefølge fra toppen. Underveis kan du gjerne endre bounding box, kategorier eller H3-oppløsning for å se hvordan resultatet endrer seg. Helt til slutt finner du en liten **CTF (Capture The Flag)**-oppgave hvor du skal grave i dataene for å finne noen skjulte "skatter" 🚩

---
### 🏷️ Overnattingskategorier vi starter med

| Kategori (Overture-kode) | Beskrivelse |
|---|---|
| `hotel` | Hotell |
| `bed_and_breakfast` | Bed & Breakfast |
| `hostel` | Vandrerhjem |
| `motel` | Motell |
| `resort` | Resort / feriesenter |
| `beach_resort` | Strandresort |
| `guest_house` | Gjestehus |
| `inn` | Vertshus |
| `lodge` | Lodge / fjellhotell |
| `cottage` | Hytte |
| `holiday_rental_home` | Feriehus (utleie) |
| `service_apartments` | Serviceapartment |
| `self_catering_accommodation` | Selvhushold-losji |
| `apartments` | Apartment |
| `campground` | Campingplass |
| `rv_park` | Bobilcamp |
| `country_house` | Landsted |
| `cabin` | Kabin |

Dette er bare 18 av over **2000 kategorier** i Overture sin taksonomi for steder (places) – i steg 3 viser vi hvordan du enkelt kan bytte disse ut med noe helt annet, f.eks. restauranter, butikker eller severdigheter.

## ⚙️ Steg 1: Oppsett

Før vi kan analysere noe som helst må vi installere pakkene vi trenger, sette opp en DuckDB-database med støtte for romlige spørringer og S3, og spørre Overture Maps sin STAC-katalog om hvilken versjon av datasettet som er nyest akkurat nå.

| Pakke | Brukes til |
|---|---|
| `duckdb` | Kjøre SQL direkte mot store Parquet-filer i skyen |
| `geopandas` | Geografisk dataanalyse i Python |
| `h3` | Sekskantet rutenett for tetthetsaggregering |
| `folium` | Interaktive kart i notebooken |
| `lonboard` / `mapclassify` | Rask visualisering av store datasett |

**Hvorfor spørre etter versjon?** Overture Maps gir ut en ny **versjon** av datasettet omtrent hver måned (f.eks. `2026-08-19.0`). Å hardkode et versjonsnummer i koden gjør notebooken utdatert etter kort tid – i stedet bruker vi DuckDB til å lese JSON direkte fra Overture sin **STAC-katalog** (SpatioTemporal Asset Catalog, [stac.overturemaps.org](https://stac.overturemaps.org/)), som alltid peker på siste versjon. Les mer i [Overture Has Fully Embraced STAC](https://docs.overturemaps.org/blog/2026/02/11/stac/).

> ⏳ Installasjonen kan ta 1–2 minutter, og må gjøres på nytt hver gang du starter en ny Colab-sesjon.

In [1]:
%%capture
%pip install duckdb geopandas h3 folium lonboard matplotlib mapclassify shapely fsspec

In [2]:
import os
import duckdb
import geopandas as gpd
import pandas as pd
import h3
import folium
import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from shapely.geometry import shape, mapping, Polygon

os.makedirs('./tmp', exist_ok=True)

# Koble opp DuckDB med romlig- og S3-støtte
conn = duckdb.connect(':memory:')
conn.execute("INSTALL spatial; LOAD spatial;")
conn.execute("SET s3_region='us-west-2';")

# Spør STAC-katalogen til Overture Maps om hvilken versjon som er nyest akkurat nå
latest_sql = "SELECT latest FROM 'https://stac.overturemaps.org/catalog.json';"
latest_version = conn.sql(latest_sql).fetchone()[0]

print(f"DuckDB versjon: {duckdb.__version__}")
print(f"H3 versjon: {h3.__version__}")
print(f"Nyeste versjon av Overture Maps: {latest_version}")

DuckDB versjon: 1.5.0
H3 versjon: 4.4.2
Nyeste versjon av Overture Maps: 2026-08-19.0


## 🗺️ Steg 2: Definer studieområdet

Gran Canaria bounding box (WGS84): `xmin=-15.85, ymin=27.70, xmax=-15.35, ymax=28.20`

> 🔧 **Prøv selv:** Vil du analysere et annet område? Finn koordinatene til et sted du kjenner med [bboxfinder.com](https://bboxfinder.com/) og bytt ut `bbox` under.

In [3]:
# Gran Canaria bounding box [xmin, ymin, xmax, ymax]
bbox = [-15.85, 27.70, -15.35, 28.20]
xmin, ymin, xmax, ymax = bbox

print(f"Studieområde: xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax}")

# Forhåndsvisning av bounding box på kart
bbox_poly = Polygon([
    (xmin, ymin), (xmax, ymin), (xmax, ymax), (xmin, ymax), (xmin, ymin)
])
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_poly], crs="EPSG:4326")
bbox_gdf.explore(color="red", style_kwds={"fillOpacity": 0.1})

Studieområde: xmin=-15.85, ymin=27.7, xmax=-15.35, ymax=28.2


## 🏨 Steg 3: Hent overnattingssteder fra Overture Maps

Nå kombinerer vi alt vi har forberedt – **nyeste versjon** (steg 1), **bounding box** (steg 2) og en liste med **kategorier** vi vil filtrere på – i én SQL-spørring mot Overture Maps sin Places-data på S3.

Overture Places har en taksonomi (kategorisystem) med over **2000 kategorier**, organisert hierarkisk under temaer som overnatting, mat & drikke, handel og severdigheter. Tabellen i toppen av notebooken viser de 18 overnattingskategoriene vi starter med. **Hive-partitionering** gjør at `*` i filstien automatisk plukker opp alle relevante filer, og **predicate pushdown** på `bbox`-kolonnene gjør at DuckDB kun leser de bitene av datasettet som faktisk kan overlappe området vårt – kilobytes, ikke terabytes.

Til slutt lagrer vi resultatet lokalt som en Parquet-fil, slik at vi kan jobbe videre med det uten å spørre S3 på nytt.

> 🔧 **Prøv selv – utforsk et tema du finner spennende!**
> Bytt ut listen `accommodation_categories` under med kategorier fra et helt annet tema – f.eks. restauranter (`restaurant`), bakerier (`bakery`), severdigheter (`tourist_attraction`) eller klatrefelt (`climbing_area`). Kjør deretter cellene under på nytt.
>
> - 🔎 [Taxonomy Browser](https://docs.overturemaps.org/guides/places/taxonomy-browser/) – søk og bla gjennom hele kategorihierarkiet interaktivt
> - 📄 [Schema-referanse for `place`](https://docs.overturemaps.org/schema/reference/places/place/) – se alle felt et sted kan ha (navn, kategori, confidence, adresse …)
> - 🗂️ Overture har flere temaer enn `places`, blant annet `buildings`, `transportation`, `base`, `divisions` og `addresses` – se [docs.overturemaps.org/schema](https://docs.overturemaps.org/schema/) for en oversikt. Samme oppskrift (STAC → bbox → filter) fungerer for alle temaene, men kolonnenavnene er litt annerledes (f.eks. `class`/`subtype` i stedet for `categories`).

In [4]:
# 🔧 Endre denne listen for å utforske andre kategorier/temaer!
accommodation_categories = [
    'hotel',
    'bed_and_breakfast',
    'hostel',
    'motel',
    'resort',
    'beach_resort',
    'guest_house',
    'inn',
    'lodging',
    'cottage',
    'holiday_rental_home',
    'service_apartments',
    'self_catering_accommodation',
    'apartments',
    'campground',
    'rv_park',
    'country_house',
    'cabin',
]

categories_sql = ', '.join(f"'{c}'" for c in accommodation_categories)
print(f"Søker etter {len(accommodation_categories)} kategorier: {accommodation_categories}")

Søker etter 18 kategorier: ['hotel', 'bed_and_breakfast', 'hostel', 'motel', 'resort', 'beach_resort', 'guest_house', 'inn', 'lodging', 'cottage', 'holiday_rental_home', 'service_apartments', 'self_catering_accommodation', 'apartments', 'campground', 'rv_park', 'country_house', 'cabin']


In [5]:
query = f"""
SELECT
    id,
    names.primary                      AS name,
    confidence,
    basic_category,
    taxonomy,
    categories.primary                 AS category,
    CAST(categories AS JSON)           AS categories_json,
    ST_X(geometry)                     AS lon,
    ST_Y(geometry)                     AS lat,
    ST_AsWKB(geometry)                 AS geometry
FROM
    read_parquet(
        's3://overturemaps-us-west-2/release/{latest_version}/theme=places/type=place/*',
        filename=true,
        hive_partitioning=1
    )
WHERE
    basic_category IN ({categories_sql})
    AND bbox.xmin BETWEEN {xmin} AND {xmax}
    AND bbox.ymin BETWEEN {ymin} AND {ymax}
"""

print(f"Spør Overture Maps ({latest_version}) på S3 – dette kan ta 30–60 sekunder ...")
result = conn.sql(query)
df_raw = result.df()

print(f"\nFant {len(df_raw)} overnattingssteder på Gran Canaria.")
df_raw.head()

Spør Overture Maps (2026-08-19.0) på S3 – dette kan ta 30–60 sekunder ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Fant 1277 overnattingssteder på Gran Canaria.


,id,name,confidence,basic_category,taxonomy,category,categories_json,lon,lat,geometry
0,5243da4f-5412-43eb-93e5-f28a70332e95,Salt Apartments Vistalmar,0.891587,lodging,"{'primary': 'service_apartment', 'hierarchy': ...",service_apartments,"{""primary"":""service_apartments"",""alternate"":[""...",-15.554668,27.765476,"[1, 1, 0, 0, 0, 61, 254, 160, 104, 253, 27, 47..."
1,01b83583-279d-44fe-8fb0-d1a801c1fb47,Bungalows Corinto 2,0.993781,hotel,"{'primary': 'hotel', 'hierarchy': ['lodging', ...",hotel,"{""primary"":""hotel"",""alternate"":[""bed_and_break...",-15.554741,27.765666,"[1, 1, 0, 0, 0, 44, 128, 41, 3, 7, 28, 47, 192..."
2,ae385c31-543a-418f-bcfb-ec917883b381,Veril Playa,0.997747,hotel,"{'primary': 'hotel', 'hierarchy': ['lodging', ...",hotel,"{""primary"":""hotel"",""alternate"":[""accommodation...",-15.557151,27.765112,"[1, 1, 0, 0, 0, 115, 151, 237, 216, 66, 29, 47..."
3,0f784908-ceef-467a-ae36-80ebf715c4e8,Apartahotel Playa Del Inglés,0.634116,hotel,"{'primary': 'hotel', 'hierarchy': ['lodging', ...",hotel,"{""primary"":""hotel"",""alternate"":null}",-15.557063,27.765011,"[1, 1, 0, 0, 0, 139, 238, 176, 86, 55, 29, 47,..."
4,fde7398e-95e1-4caf-ba79-913733109fca,Hotel Beverly Park,0.770000,hotel,"{'primary': 'hotel', 'hierarchy': ['lodging', ...",hotel,"{""primary"":""hotel"",""alternate"":null}",-15.557931,27.764474,"[1, 1, 0, 0, 0, 77, 52, 201, 35, 169, 29, 47, ..."


In [6]:
conn.register('accommodation_raw', df_raw)
conn.execute("""
    COPY (
        SELECT id, name, confidence, category, categories_json, lon, lat
        FROM accommodation_raw
    ) TO './tmp/gran_canaria_accommodation.parquet' (FORMAT PARQUET)
""")
print("Lagret til ./tmp/gran_canaria_accommodation.parquet")

Lagret til ./tmp/gran_canaria_accommodation.parquet


## 🔷 Steg 4: Tetthetsanalyse med H3-hexagoner

**H3** deler jordoverflaten inn i sekskantede celler i forskjellige oppløsninger. Vi bruker det til å (1) gi hvert overnattingssted en H3-celleindeks, (2) telle antall steder per celle – totalt og per kategori, (3) konvertere cellene til polygoner, og (4) tegne resultatet som et fargelagt tetthetskart.

H3-oppløsning for Gran Canaria:
- **Oppløsning 7** (~5.2 km² / celle) – oversikt over hele øya
- **Oppløsning 8** (~0.7 km² / celle) – nabolagsnivå ✅ brukes som standard
- **Oppløsning 9** (~0.1 km² / celle) – svært finmasket

> 🔧 Endre `H3_RESOLUTION` under for å utforske andre oppløsninger.

Til slutt eksporterer vi kartet som en HTML-fil du kan laste ned og åpne i nettleseren, eller dele med andre.

In [7]:
H3_RESOLUTION = 8  # Endre til 7 eller 9 for å utforske andre oppløsninger

# Fjern rader uten koordinater og tildel H3-celleindeks
df = df_raw.dropna(subset=['lat', 'lon']).copy()
df['h3_index'] = df.apply(
    lambda row: h3.latlng_to_cell(row['lat'], row['lon'], H3_RESOLUTION),
    axis=1
)

# Totalt antall per H3-celle
hex_density = df.groupby('h3_index').size().reset_index(name='total_count')

# Kategori-fordeling per H3-celle (pivot)
category_pivot = (
    df.groupby(['h3_index', 'category'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
hex_density = hex_density.merge(category_pivot, on='h3_index', how='left')

def h3_to_polygon(h3_index):
    """Konverterer en H3-celleindeks til et Shapely-polygon."""
    boundary = h3.cell_to_boundary(h3_index)  # liste av (lat, lon)-par
    return Polygon([(lon, lat) for lat, lon in boundary])  # shapely vil ha (lon, lat)

# Senterkoordinater og polygon-geometri for hver H3-celle
hex_density['center_lat'] = hex_density['h3_index'].apply(lambda x: h3.cell_to_latlng(x)[0])
hex_density['center_lon'] = hex_density['h3_index'].apply(lambda x: h3.cell_to_latlng(x)[1])
hex_density['geometry'] = hex_density['h3_index'].apply(h3_to_polygon)

gdf_hex = gpd.GeoDataFrame(hex_density, geometry='geometry', crs='EPSG:4326')

print(f"H3-oppløsning: {H3_RESOLUTION}")
print(f"Antall unike H3-celler: {len(gdf_hex)}")
gdf_hex[['h3_index', 'total_count', 'geometry']].sort_values('total_count', ascending=False).head(10)

H3-oppløsning: 8
Antall unike H3-celler: 320


,h3_index,total_count,geometry
256,88346b4c49fffff,61,"POLYGON ((-15.43273 28.14581, -15.43778 28.143..."
53,88344ca5edfffff,53,"POLYGON ((-15.57704 27.75993, -15.58207 27.757..."
99,88344caf55fffff,52,"POLYGON ((-15.7118 27.79719, -15.71683 27.7945..."
52,88344ca5e9fffff,52,"POLYGON ((-15.56704 27.76002, -15.57207 27.757..."
34,88344ca517fffff,47,"POLYGON ((-15.57197 27.76767, -15.57699 27.765..."
32,88344ca513fffff,36,"POLYGON ((-15.56197 27.76775, -15.56699 27.765..."
41,88344ca533fffff,32,"POLYGON ((-15.58705 27.75985, -15.59207 27.757..."
252,88346b4c41fffff,32,"POLYGON ((-15.43783 28.13807, -15.44288 28.135..."
246,88346b4c19fffff,32,"POLYGON ((-15.41811 28.10741, -15.42316 28.104..."
298,88346b4e23fffff,32,"POLYGON ((-15.42762 28.15354, -15.43267 28.150..."


In [8]:
known_categories = [
    'hotel', 'bed_and_breakfast', 'hostel', 'motel', 'resort', 'beach_resort',
    'guest_house', 'inn', 'lodge', 'cottage', 'holiday_rental_home',
    'service_apartments', 'self_catering_accommodation', 'apartments',
    'campground', 'rv_park', 'country_house', 'cabin'
]
tooltip_cats = [c for c in known_categories if c in gdf_hex.columns]
tooltip_fields = ['total_count', 'h3_index'] + tooltip_cats
tooltip_aliases = ['Totalt', 'H3-celle'] + tooltip_cats

center_lat = (ymin + ymax) / 2
center_lon = (xmin + xmax) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles='Esri.WorldGrayCanvas'
)

folium.Choropleth(
    geo_data=gdf_hex[['h3_index', 'total_count', 'geometry']].to_json(),
    name='Overnattingstetthet',
    data=gdf_hex[['h3_index', 'total_count']],
    columns=['h3_index', 'total_count'],
    key_on='feature.properties.h3_index',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name=f'Overnattingssteder per H3-celle (oppløsning {H3_RESOLUTION})',
    nan_fill_color='transparent',
).add_to(m)

folium.GeoJson(
    data=gdf_hex.to_json(),
    name='Detaljer',
    style_function=lambda x: {'fillOpacity': 0, 'weight': 0},
    tooltip=folium.GeoJsonTooltip(
        fields=tooltip_fields,
        aliases=tooltip_aliases,
        localize=True,
    )
).add_to(m)

folium.LayerControl().add_to(m)

# Eksporter kartet som en HTML-fil
html_path = './tmp/gran_canaria_tetthetskart.html'
m.save(html_path)
print(f"Kart lagret til {html_path}")

try:
    from google.colab import files
    files.download(html_path)
except ImportError:
    print("Kjører ikke i Colab - finn filen lokalt i ./tmp/")

m

Kart lagret til ./tmp/gran_canaria_tetthetskart.html
Kjører ikke i Colab - finn filen lokalt i ./tmp/


## 🌍 Steg 5: Eksporter GeoJSON og bygg ditt eget kart

Vi eksporterer punktdataene (`df`) og H3-tetthetsdataene (`gdf_hex`) som GeoJSON - et åpent, tekstbasert format som de fleste kartbiblioteker (MapLibre GL JS, Leaflet, Mapbox ...) kan lese direkte.

Last ned de to filene under, og bruk dem som utgangspunkt for et eget prosjekt: be en språkmodell eller kodeagent (f.eks. Claude Code, Cursor eller GitHub Copilot) om å bygge et interaktivt kart med **MapLibre GL JS**.

> 💬 **Eksempel-instruks du kan gi en kodeagent:**
>
> "Lag en enkel HTML-side med MapLibre GL JS som viser to lag: `gran_canaria_overnatting_punkter.geojson` som klikkbare punkter, og `gran_canaria_overnatting_h3.geojson` som et fargelagt tetthetskart (choropleth) basert på feltet `total_count`. Bruk en gratis bakgrunnskart-stil (f.eks. OpenFreeMap eller MapTiler sin demo-stil), og legg til en forklaring (legend) for fargeskalaen."

In [18]:
points_path = './tmp/gran_canaria_overnatting_punkter.geojson'
hex_path = './tmp/gran_canaria_overnatting_h3.geojson'

gdf_points = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['lon'], df['lat']),
    crs='EPSG:4326'
)
gdf_points[['id', 'name', 'category', 'h3_index', 'geometry']].to_file(points_path, driver='GeoJSON')
gdf_hex.to_file(hex_path, driver='GeoJSON')

print(f"Lagret {len(gdf_points)} punkter til {points_path}")
print(f"Lagret {len(gdf_hex)} H3-celler til {hex_path}")

try:
    from google.colab import files
    files.download(points_path)
    files.download(hex_path)
except ImportError:
    print('Kjører ikke i Colab - finn filene lokalt i ./tmp/')

Lagret 1277 punkter til ./tmp/gran_canaria_overnatting_punkter.geojson
Lagret 320 H3-celler til ./tmp/gran_canaria_overnatting_h3.geojson
Kjører ikke i Colab - finn filene lokalt i ./tmp/


## 💡 Refleksjonsspørsmål

1. **Hvor er turist-hotspotene?** Stemmer de tetteste cellene med kjente reisemål (Playa del Inglés, Maspalomas, Las Palmas)?
2. **Hoteller vs. apartments**: Er korttidsutleie-markedet (apartments, feriehus) mer spredt eller mer konsentrert enn hoteller? Du kan regne dette ut selv med kategori-kolonnene i `gdf_hex` fra steg 4.
3. **H3-oppløsning**: Prøv å endre `H3_RESOLUTION` i steg 4 til 7 eller 9 og kjør cellene på nytt – hvordan endrer historien kartet forteller seg?
4. **Datakvalitet**: Hvor sikker er du på at Overture Maps-dataene er komplette for en spansk øy? Hvordan kunne du kryssvalidert dette?
5. **Neste steg**: Kunne du lagt til befolkningsdata, strender eller veinettverk for å forklare tetthetsmønstrene?

## 🚩 CTF: Skattejakt i dataene!

Nå setter vi kunnskapen din på prøve! Under følger tre oppgaver hvor du skal grave i dataene for å finne helt spesifikke enkeltobjekter – litt som et miniatyr **Capture The Flag (CTF)**.

Overture Maps oppdateres kontinuerlig, så de eksakte svarene endrer seg over tid. Derfor beregner **sjekke-cellene** det korrekte svaret på nytt hver gang du kjører notebooken – løsningen er alltid riktig for *din* kjøring, uansett når du gjør den.

**Slik gjør du det for hvert flagg:**
1. Les oppgaven.
2. Skriv din egen kode i en ny celle for å utforske og finne svaret (bruk `df`, `df_raw` eller `gdf_hex` som allerede er definert over).
3. Fyll inn svaret ditt i variabelen merket 🚩 i sjekke-cellen, og kjør den for å se om du har rett.

### 🚩 Flagg 1 — Det lengste navnet

Blant alle overnattingsstedene i `df_raw`: finn stedet med det (tegnmessig) **lengste navnet** (`name`-kolonnen). Hva heter det?

💡 **Tips:** `.str.len()` på en pandas-tekstkolonne gir lengden på hver streng, og `.idxmax()` gir indeksen til den høyeste verdien.

In [10]:
# ✍️ Skriv din egen utforskende kode her for å finne stedet med lengst navn


In [11]:
# ✅ Sjekk svaret på Flagg 1
flagg1_svar = ""  # 🚩 Skriv inn navnet du fant her

navn_lengder = df_raw['name'].dropna()
riktig_navn = navn_lengder.loc[navn_lengder.str.len().idxmax()]

if flagg1_svar.strip().lower() == riktig_navn.strip().lower():
    print(f"✅ Riktig! Flagget er: {riktig_navn}")
else:
    print("❌ Ikke helt riktig ennå. Prøv igjen!")

❌ Ikke helt riktig ennå. Prøv igjen!


### 🚩 Flagg 2 — Den tetteste campingcellen

Blant H3-cellene i `gdf_hex` (oppløsning 8): finn cellen (`h3_index`) med **flest campingrelaterte overnattingssteder** – det vil si summen av kategoriene `campground` og `rv_park`. Hvor mange slike steder ligger i denne cellen?

💡 **Tips:** Summer de to kolonnene radvis med `.sum(axis=1)`, og finn raden med høyest verdi med `.idxmax()`.

In [12]:
# ✍️ Skriv din egen utforskende kode her for å finne campingcellen


In [13]:
# ✅ Sjekk svaret på Flagg 2
flagg2_celle = ""    # 🚩 Skriv inn h3_index du fant her
flagg2_antall = 0    # 🚩 Skriv inn antallet camping-steder i cellen

camping_kategorier = [c for c in ['campground', 'rv_park'] if c in gdf_hex.columns]
camping_total = gdf_hex[camping_kategorier].sum(axis=1) if camping_kategorier else pd.Series(dtype=int)
riktig_rad = gdf_hex.loc[camping_total.idxmax()]
riktig_celle = riktig_rad['h3_index']
riktig_antall = int(camping_total.max())

if flagg2_celle == riktig_celle and int(flagg2_antall) == riktig_antall:
    print(f"✅ Riktig! Celle {riktig_celle} har {riktig_antall} campingrelaterte steder.")
else:
    print("❌ Ikke helt riktig ennå. Prøv igjen!")

❌ Ikke helt riktig ennå. Prøv igjen!


### 🚩 Flagg 3 (bonus) — Nærmest et kjent landemerke

**Maspalomas fyr** (Faro de Maspalomas) ligger på `lat=27.7378, lon=-15.5893`. Hvilket overnattingssted i `df` ligger **nærmest** fyret – og hvor mange meter unna er det (avrundet til nærmeste 10 meter)?

💡 **Tips:** Bruk Haversine-formelen under til å beregne avstanden (i meter) mellom fyret og hvert overnattingssted, og finn stedet med kortest avstand.

In [14]:
from math import radians, sin, cos, sqrt, atan2

def haversine_meter(lat1, lon1, lat2, lon2):
    """Avstand i meter mellom to lat/lon-punkter (kloden tilnærmet som en kule)."""
    R = 6_371_000
    phi1, phi2 = radians(lat1), radians(lat2)
    delta_phi = radians(lat2 - lat1)
    delta_lambda = radians(lon2 - lon1)
    a = sin(delta_phi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(delta_lambda / 2) ** 2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

fyr_lat, fyr_lon = 27.7378, -15.5893

In [15]:
# ✍️ Skriv din egen utforskende kode her – bruk haversine_meter() til å finne nærmeste sted


In [16]:
# ✅ Sjekk svaret på Flagg 3
flagg3_navn = ""       # 🚩 Skriv inn navnet på stedet du fant her
flagg3_avstand_m = -1  # 🚩 Skriv inn avstanden i meter (avrundet til nærmeste 10 meter)

avstander = df.apply(lambda r: haversine_meter(fyr_lat, fyr_lon, r['lat'], r['lon']), axis=1)
riktig_indeks = avstander.idxmin()
riktig_navn = df.loc[riktig_indeks, 'name']
riktig_avstand = round(avstander.min() / 10) * 10

navn_ok = flagg3_navn.strip().lower() == str(riktig_navn).strip().lower()
avstand_ok = abs(flagg3_avstand_m - riktig_avstand) <= 10

if navn_ok and avstand_ok:
    print(f"✅ Riktig! {riktig_navn} ligger ca. {riktig_avstand} meter fra Maspalomas fyr.")
else:
    print("❌ Ikke helt riktig ennå. Prøv igjen!")

❌ Ikke helt riktig ennå. Prøv igjen!


## 🎉 Gratulerer!

Hvis du fant alle tre flaggene har du vist at du kan navigere, filtrere og analysere store geografiske datasett med DuckDB, pandas og H3 – helt uten å laste ned mer data enn nødvendig. Del funnene dine (og eventuelt egne kategorivalg fra steg 3!) med resten av klassen.